In [34]:
import numpy as np
import pandas as pd
import seaborn as sns
import cv2,os
import matplotlib.pyplot as plt
from keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential, load_model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import InceptionResNetV2
from keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Flatten, Dense, Dropout,BatchNormalization

In [10]:
os.environ["TF_ENABLE_ONEDNN_OPTS"] = '0'
TRAIN_DIR = "./data_tf/Train/"
TEST_DIR = "./data_tf/Test/"

In [9]:
train_gen = ImageDataGenerator(rescale=1/255,
                              horizontal_flip=True,
                              shear_range=0.4,
                              zoom_range=0.2,
                              vertical_flip=True)

test_gen = ImageDataGenerator(rescale=1/255,
                              horizontal_flip=True,
                              shear_range=0.4,
                              zoom_range=0.2,
                              vertical_flip=True)

In [12]:
train_data  = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=(300,300),
    class_mode="categorical"
)

test_data  = test_gen.flow_from_directory(
    TEST_DIR,
    target_size=(300,300),
    class_mode="categorical"
)

Found 10500 images belonging to 7 classes.
Found 1200 images belonging to 7 classes.


In [18]:
x,y = next(train_data)
x.shape 

(32, 300, 300, 3)

In [21]:
base_model =  InceptionResNetV2(
    include_top=False,
    input_shape=(300,300,3),
    weights="imagenet"
)
base_model.trainable =  False



219055592/219055592 [==============================] - 23s 0us/step


In [35]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dense(512,activation="relu"),
    Dense(256,activation="relu"),
    Dropout(0.2),
    Dense(7,activation="softmax")
])

In [36]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 inception_resnet_v2 (Funct  (None, 8, 8, 1536)        54336736  
 ional)                                                          
                                                                 
 global_average_pooling2d_2  (None, 1536)              0         
  (GlobalAveragePooling2D)                                       
                                                                 
 batch_normalization_203 (B  (None, 1536)              6144      
 atchNormalization)                                              
                                                                 
 dense_3 (Dense)             (None, 512)               786944    
                                                                 
 dense_4 (Dense)             (None, 256)               131328    
                                                      

In [37]:
model.compile(loss="categorical_crossentropy",optimizer="adam",metrics=["acc"])

In [38]:
callbacks =  EarlyStopping(patience=10,restore_best_weights=True)

In [ ]:
history = model.fit(train_data,validation_data=test_data,epochs=15,callbacks=[callbacks])

Epoch 1/15
 38/329 [==>...........................] - ETA: 32:38 - loss: 0.9618 - acc: 0.7064

C:\Users\RISHABH\AppData\Roaming\Python\Python310\site-packages\PIL\Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


132/329 [===========>..................] - ETA: 22:37 - loss: 0.7574 - acc: 0.7694